In [1]:
import os
import torch
import numpy as np
import tifffile
import pandas as pd
from cellpose import models, utils
from skimage.measure import regionprops, label
from skimage.draw import ellipse_perimeter
from skimage.morphology import binary_erosion
from skimage.measure import moments_central, moments_hu
from tqdm import tqdm 
import cv2
torch.set_float32_matmul_precision('high')

KeyboardInterrupt: 

In [ ]:
# Parameters
input_folder = r"L:\43-RVZ\AIMicroscopy\Mitarbeiter\2_Data\1_NikonTi2\FK_P001_EX005_2025_03_21 60xWIA 1.5x CRISPRi mCh\TIF\results\stacked"
output_folder = os.path.join(input_folder, "cyto3")  # Create results folder dynamically based on model name
os.makedirs(output_folder, exist_ok=True)  # Ensure output folder exists

diameter = None  # Or set to expected cell size
min_size = 35  # Minimum size of a segmented object in pixels to keep
use_gpu = True  # Enable GPU acceleration if available
# Initialize Cellpose model with GPU support if available
model = models.Cellpose(gpu=True, model_type='cyto3')

In [ ]:
# Utility function: shape features
def shape_features(region):
    area = region.area
    perimeter = region.perimeter if region.perimeter > 0 else 1
    roundness = 4 * np.pi * area / (perimeter ** 2)
    major_axis = region.major_axis_length
    minor_axis = region.minor_axis_length
    aspect_ratio = major_axis / minor_axis if minor_axis != 0 else 0
    return {
        "Area": area,
        "Length": major_axis,
        "Width": minor_axis,
        "Roundness": roundness,
    }

In [ ]:
# Analyze a single stack and return results as a DataFrame
def analyze_single_stack(stack_path):
    # Load TIFF stack
    stack = tifffile.imread(stack_path)
    dic_img = stack[0]   # DIC channel for segmentation
    fluo_img = stack[1]  # Fluorescent channel for measurement

    # Normalize DIC image for better segmentation
    dic_img = (dic_img - np.min(dic_img)) / (np.max(dic_img) - np.min(dic_img))

    # Run Cellpose segmentation on DIC channel with GPU acceleration if available
    masks, flows, styles, diams = model.eval(dic_img, diameter=diameter, channels=[0, 0])

    # Extract region properties using fluorescent image as intensity reference
    labeled = label(masks)
    props = regionprops(labeled, intensity_image=fluo_img)

    data = []
    for prop in props:
        area = prop.area
        length = prop.major_axis_length
        width = prop.minor_axis_length
        roundness = (
            4 * np.pi * area / (prop.perimeter ** 2) if prop.perimeter != 0 else 0
        )
        mean_intensity = prop.mean_intensity
        std_intensity = np.std(fluo_img[prop.coords[:, 0], prop.coords[:, 1]])
        data.append(
            {
                "Label": prop.label,
                "Area": area,
                "Length": length,
                "Width": width,
                "Aspect Ratio": length / width if width != 0 else 0,
                "Roundness": roundness,
                "Mean Intensity": mean_intensity,
                "Std Intensity": std_intensity,
                "Centroid X": prop.centroid[1],
                "Centroid Y": prop.centroid[0],
                "Orientation": prop.orientation,
            }
        )

    return pd.DataFrame(data), masks

In [ ]:
# Loop through all TIFF files and process them with a progress bar
tiff_files = [f for f in os.listdir(input_folder) if f.endswith(".tif") or f.endswith(".tiff")]
print(f"Found {len(tiff_files)} TIFF files to process.")

for fname in tqdm(tiff_files, desc="Processing stacks"):
    fpath = os.path.join(input_folder, fname)
    
    try:
        # Analyze single stack and save results
        df, masks = analyze_single_stack(fpath)

        # Save results to Excel file in the output folder
        excel_output_path = os.path.join(output_folder, f"{os.path.splitext(fname)[0]}.xlsx")
        df.to_excel(excel_output_path, index=False)

        # Save masks as a TIFF file in the output folder
        mask_output_path = os.path.join(output_folder, f"{os.path.splitext(fname)[0]}_masks.tif")
        tifffile.imwrite(mask_output_path, masks.astype(np.uint16))

    except Exception as e:
        print(f"⚠️ Error processing {fname}: {e}")

print("✅ Batch analysis complete.")